# BioAI Hub — Layer 2: Primera CNN sobre Chest X-Ray

En esta layer descargo el dataset Chest X-Ray de Kaggle, exploro la distribución de clases, construyo el DataLoader con augmentación, defino la arquitectura BioAICNN, y entreno el modelo desde cero.

**Dataset:** [Chest X-Ray (Pneumonia) — Kaggle](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)  
~5,800 imágenes, 2 clases: NORMAL / PNEUMONIA

## 0. Setup

Monto Drive y verifico GPU antes de cualquier otra cosa. Si el runtime se reinicia entre sesiones, esta celda restaura el entorno.

In [ ]:
import os
import torch
from google.colab import drive

drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/bioai-colab'
DATA_DIR = '/content/data'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Descargar el dataset

Uso la API de Kaggle para descargar directamente en Colab. La celda de abajo pide que subas el archivo `kaggle.json` — es el token que generaste en kaggle.com → Settings → API.

In [ ]:
from google.colab import files

print("Subí el archivo kaggle.json cuando aparezca el botón...")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Credenciales configuradas")

In [ ]:
# La descarga tarda ~3-5 minutos (~1.3 GB)
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p {DATA_DIR} --unzip
print("Dataset descargado")

In [ ]:
# Verificar estructura de carpetas
chest_dir = os.path.join(DATA_DIR, 'chest_xray')

for split in ['train', 'val', 'test']:
    split_path = os.path.join(chest_dir, split)
    if os.path.exists(split_path):
        for clase in os.listdir(split_path):
            n = len(os.listdir(os.path.join(split_path, clase)))
            print(f"{split}/{clase}: {n} imágenes")

## 2. Exploración del dataset

Antes de entrenar vale la pena mirar los datos. El dataset tiene un desbalanceo importante: hay casi 3x más imágenes de PNEUMONIA que de NORMAL. Si no lo compenso, el modelo puede aprender a predecir siempre PNEUMONIA y tener un 75% de accuracy sin realmente aprender nada útil.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

train_dir = Path(chest_dir) / 'train'

conteo = {}
for clase in ['NORMAL', 'PNEUMONIA']:
    conteo[clase] = len(list((train_dir / clase).glob('*.jpeg')) +
                        list((train_dir / clase).glob('*.jpg')) +
                        list((train_dir / clase).glob('*.png')))

total = sum(conteo.values())
print("Distribución en training set:")
for clase, n in conteo.items():
    print(f"  {clase}: {n} imágenes ({n/total*100:.1f}%)")

In [ ]:
import random

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Chest X-Ray — muestras del training set', fontsize=13)

for row, clase in enumerate(['NORMAL', 'PNEUMONIA']):
    imagenes = list((train_dir / clase).glob('*.jpeg'))[:4]
    for col, img_path in enumerate(imagenes):
        img = mpimg.imread(img_path)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].set_title(clase, fontsize=10)
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## 3. DataLoader

Para el training set uso augmentación básica (flip horizontal y rotación leve). Las radiografías de tórax no se invierten verticalmente ni tienen cambios de color relevantes, así que limito la augmentación a transformaciones anatómicamente plausibles.

La normalización con los valores de ImageNet es estándar aunque las radiografías sean en escala de grises — los modelos pre-entrenados esperan esos valores y funciona bien en la práctica.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(str(train_dir),       transform=transform_train)
val_ds   = datasets.ImageFolder(str(Path(chest_dir) / 'val'),  transform=transform_eval)
test_ds  = datasets.ImageFolder(str(Path(chest_dir) / 'test'), transform=transform_eval)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Clases: {train_ds.classes}")
print(f"Train:  {len(train_ds)} imágenes")
print(f"Val:    {len(val_ds)} imágenes")
print(f"Test:   {len(test_ds)} imágenes")

In [ ]:
# Pesos para compensar el desbalanceo en la loss function
# La clase minoritaria recibe más penalización cuando el modelo se equivoca
n_normal    = conteo['NORMAL']
n_pneumonia = conteo['PNEUMONIA']
n_total     = n_normal + n_pneumonia

# clase 0 = NORMAL, clase 1 = PNEUMONIA (orden alfabético de ImageFolder)
class_weights = torch.tensor([
    n_total / (2 * n_normal),
    n_total / (2 * n_pneumonia)
]).to(device)

print(f"Peso NORMAL:    {class_weights[0]:.3f}")
print(f"Peso PNEUMONIA: {class_weights[1]:.3f}")

## 4. Arquitectura BioAICNN

Red convolucional propia con 3 bloques de extracción de features seguidos de un clasificador. Es la arquitectura que uso en producción en la app de BioAI Hub.

Cada bloque sigue el patrón Conv → BatchNorm → ReLU → MaxPool. BatchNorm estabiliza el entrenamiento normalizando las activaciones por batch. MaxPool reduce la resolución espacial a la mitad en cada bloque, lo que reduce el costo computacional y fuerza al modelo a aprender representaciones más abstractas.

In [ ]:
import torch.nn as nn

class BioAICNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            # bloque 1: 3 → 32 canales, 224x224 → 112x112
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # bloque 2: 32 → 64 canales, 112x112 → 56x56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # bloque 3: 64 → 128 canales, 56x56 → 28x28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = BioAICNN().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros totales: {total_params:,}")

## 5. Entrenamiento

Uso Adam con lr=1e-3 y CrossEntropyLoss con los pesos calculados arriba. Guardo el mejor modelo en Drive según val_loss — así si la sesión cae puedo retomar sin perder el mejor checkpoint.

In [ ]:
from tqdm import tqdm

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(weight=class_weights)

EPOCHS = 10
CHECKPOINT_PATH = os.path.join(WORK_DIR, 'bioaicnn_best.pth')

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(device), labels.to(device)

            if training:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += images.size(0)

    return total_loss / total, correct / total


print(f"Entrenando {EPOCHS} epochs...\n")

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss,   val_acc   = run_epoch(val_loader,   training=False)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    mejoro = val_loss < best_val_loss
    if mejoro:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, CHECKPOINT_PATH)

    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  "
          f"val_loss={val_loss:.4f}  val_acc={val_acc:.3f}"
          f"{'  ← guardado' if mejoro else ''}")

print(f"\nMejor val_loss: {best_val_loss:.4f}")

## 6. Curvas de entrenamiento

Si val_loss sube mientras train_loss sigue bajando, el modelo está overfitting: memoriza el training set en lugar de generalizar. Con 10 epochs y Dropout(0.5) debería estar controlado.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, history['train_loss'], label='train')
ax1.plot(epochs_range, history['val_loss'],   label='val')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], label='train')
ax2.plot(epochs_range, history['val_acc'],   label='val')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('BioAICNN — curvas de entrenamiento', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'training_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

## 7. Evaluación en test set

Cargo el mejor checkpoint guardado en Drive y evalúo sobre el test set, que el modelo nunca vio durante el entrenamiento.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

# Cargar mejor checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Checkpoint cargado — epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f}")

# Inferencia sobre test set
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print("\n--- Classification Report ---")
print(classification_report(all_labels, all_preds,
                             target_names=train_ds.classes))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=train_ds.classes,
            yticklabels=train_ds.classes, ax=ax)
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión — test set')
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'confusion_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()